# 03 — Clean data and build edge list

Loads raw JSON, builds clean DataFrames for repos and edges, saves to processed/.

In [ ]:
import json
import pandas as pd
import numpy as np

In [ ]:
with open("../data/raw/repos_raw.json") as f:
    repos = json.load(f)

repos.sort(key=lambda x: x["stargazers_count"], reverse=True)
repos = repos[:500]
print(f"Working with {len(repos)} repos")

In [ ]:
rows = []
for r in repos:
    rows.append({
        "full_name": r["full_name"],
        "owner": r["owner"]["login"],
        "name": r["name"],
        "stars": r["stargazers_count"],
        "forks": r["forks_count"],
        "watchers": r["watchers_count"],
        "language": r["language"],
        "created_at": r["created_at"],
        "pushed_at": r["pushed_at"],
        "open_issues": r["open_issues_count"],
        "license": r["license"]["spdx_id"] if r["license"] else None,
        "topics": r["topics"],
        "description_length": len(r["description"]) if r["description"] else 0,
    })

df = pd.DataFrame(rows)
df["created_at"] = pd.to_datetime(df["created_at"])
df["pushed_at"] = pd.to_datetime(df["pushed_at"])
df["age_days"] = (df["pushed_at"] - df["created_at"]).dt.days

print(df.shape)
df.head()

In [ ]:
print("Missing language:", df["language"].isna().sum())
print("Missing license:", df["license"].isna().sum())
print()
print("Stars - median:", int(df["stars"].median()), "max:", df["stars"].max(), "min:", df["stars"].min())
print("Forks - median:", int(df["forks"].median()), "max:", df["forks"].max())
print()
print("Top 10 languages:")
print(df["language"].value_counts().head(10))

In [ ]:
df.to_csv("../data/processed/repos_clean.csv", index=False)
print("Saved repos_clean.csv")

In [ ]:
with open("../data/raw/contributors_raw.json") as f:
    contribs = json.load(f)

edges = []
for repo_name, contrib_list in contribs.items():
    for c in contrib_list:
        edges.append({
            "repo": repo_name,
            "contributor": c["login"],
            "contributions": c["contributions"],
        })

edges_df = pd.DataFrame(edges)
print("Edges shape:", edges_df.shape)
print("Unique repos:", edges_df["repo"].nunique())
print("Unique contributors:", edges_df["contributor"].nunique())

In [ ]:
edges_df.to_csv("../data/processed/edges.csv", index=False)
print("Saved edges.csv")